In [1]:
import pandas as pd
import numpy as np
import json
import os

# SSL loss

In [2]:
trained_data_root = '../GraPPI_data/trained_data_ssl_finetune/'
hidden_dims = [5,6,7,8,9,10] #[5-10]
num_layers = [1,2,3,4,5,6] #[1-6]
input_types = ['base', 'esm'] #['base', 'esm']
ssl_df = {'input_feat': [], 'encoder_num_module': [], 'encoder_hdim': [], 'test_loss': []}
for input_type in input_types:
	for hdim in hidden_dims:
		for nlayer in num_layers:
			folder = f"{nlayer}layers_{hdim}hdim{'_'+input_type if input_type != 'base' else ''}"
			save_dir = os.path.join(trained_data_root, folder)
			if os.path.exists(f"{save_dir}/ssl_edge_config.json"):
				with open(f"{save_dir}/ssl_edge_config.json", 'r') as f:
					data_single = json.load(f)
				ssl_df['input_feat'].append("GraPPI" if input_type == 'esm' else "GraPPI-base")
				ssl_df['encoder_num_module'].append(nlayer)
				ssl_df['encoder_hdim'].append(2**hdim)
				ssl_df['test_loss'].append(data_single['best_val_loss'])
			else:
				print(f"{save_dir}/ssl_edge_config.json not found")
				continue
ssl_df = pd.DataFrame(ssl_df)

# ∆G prediction

## ── Load pretrain+MLP ΔG results ──

In [3]:
trained_data_root = '../GraPPI_data/trained_data_ssl_finetune/'
hidden_dims = [5,6,7,8,9, 10] #[5-10]
num_layers = [1,2,3,4,5,6] #[1-6]
input_types = ['base', 'esm'] #['base', 'esm']
data_all = {}
for input_type in input_types:
	for hdim in hidden_dims:
		for nlayer in num_layers:
			if input_type == 'base':
				folder = f"{nlayer}layers_{hdim}hdim_jkmean"
			else:
				folder = f"{nlayer}layers_{hdim}hdim_{input_type}_jkmean"
			save_dir = os.path.join(trained_data_root, folder)
			if os.path.exists(f"{save_dir}/finetune_regressor_jkmean_5fold_log.json"):
				with open(f"{save_dir}/finetune_regressor_jkmean_5fold_log.json", 'r') as f:
					data_single = json.load(f)
				val_rps = [result_dict['best_val_rp'] for _,result_dict in data_single.items()]
				test_rps = [result_dict['test_rp'] for _,result_dict in data_single.items()]
				test_mae = [result_dict['test_mae'] if 'test_mae' in result_dict.keys() else None for _,result_dict in data_single.items()]
				data_all[f"{folder}"] = {'test_rp': test_rps, 'val_rp': val_rps, 'test_mae': test_mae}
			else:
				print(f"{save_dir}/finetune_regressor_jkmean_5fold_log.json not found")
				continue
dg_cv_df_jk = {}
nlayers, hdims, mean_val_rps, mean_test_rps, max_val_rps, max_test_rps, min_test_maes, mean_test_maes, models = [], [], [], [], [], [], [], [], []
test_rp_stds, val_rp_stds, test_mae_stds = [], [], []
for key, value in data_all.items():
	nlayer, hdim = int(key.split('_')[0].replace('layers','')), int(key.split('_')[1].replace('hdim',''))
	if len(key.split('_')) > 3:
		model = key.split('_')[2]
	else:
		model = 'base'
	nlayers.append(nlayer)
	hdims.append(2**hdim)
	models.append(model)
	mean_test_rps.append(np.mean(value['test_rp']))
	mean_val_rps.append(np.mean(value['val_rp']))
	max_test_rps.append(np.max(value['test_rp']))
	max_val_rps.append(np.max(value['val_rp']))
	test_rp_stds.append(np.std(value['test_rp']))
	val_rp_stds.append(np.std(value['val_rp']))
	valid_maes = [v for v in value['test_mae'] if v is not None]
	test_mae_stds.append(np.std(valid_maes) if valid_maes else None)
	min_test_maes.append(np.min(valid_maes) if valid_maes else None)
	mean_test_maes.append(np.mean(valid_maes) if valid_maes else None)

dg_cv_df_jk['input'] = models
dg_cv_df_jk['encoder_num_module'] = nlayers
dg_cv_df_jk['encoder_hdim'] = hdims
dg_cv_df_jk['model'] = 'MLP'
dg_cv_df_jk['max_test_rp'] = max_test_rps
dg_cv_df_jk['max_val_rp'] = max_val_rps
dg_cv_df_jk['mean_val_rp'] = mean_val_rps
dg_cv_df_jk['mean_test_rp'] = mean_test_rps
dg_cv_df_jk['min_test_mae'] = min_test_maes
dg_cv_df_jk['mean_test_mae'] = mean_test_maes
dg_cv_df_jk['test_rp_std'] = test_rp_stds
dg_cv_df_jk['val_rp_std'] = val_rp_stds
dg_cv_df_jk['test_mae_std'] = test_mae_stds

dg_cv_df_jk = pd.DataFrame(dg_cv_df_jk)


## ── Load baseline ΔG results ──

In [4]:
trained_data_root = '../GraPPI_data/trained_data_ssl_finetune/'

baseline_sources = {
    'base':   ('baseline',     'dG_baseline_results.json'),
    'esm':    ('esm_baseline', 'dG_esm_baseline_results.json'),
}

dg_baseline_df = {
    'input': [], 'encoder_num_module': [], 'encoder_hdim': [],'model': [],
    'max_test_rp': [], 'max_val_rp': [], 'mean_val_rp': [], 'mean_test_rp': [], 'min_test_mae': [], 'mean_test_mae': [],
    'test_rp_std': [], 'val_rp_std': [], 'test_mae_std': []
}

for input_type, (subfolder, json_file) in baseline_sources.items():
    fpath = os.path.join(trained_data_root, subfolder, json_file)
    if not os.path.exists(fpath):
        print(f"{fpath} not found")
        continue
    with open(fpath, 'r') as f:
        data = json.load(f)

    # ── NN results ──
    nn = data['nn']
    fold_keys = [k for k in nn if k != 'aggregate']
    val_rps  = [nn[k]['best_val_rp'] for k in fold_keys]
    test_rps = [nn[k]['test']['rp']  for k in fold_keys]
    test_maes = [nn[k]['test']['mae']  for k in fold_keys]

    dg_baseline_df['input'].append(f'{input_type}')
    dg_baseline_df['encoder_num_module'].append(None)
    dg_baseline_df['encoder_hdim'].append(None)
    dg_baseline_df['model'].append('MLP')
    dg_baseline_df['max_val_rp'].append(np.max(val_rps))
    dg_baseline_df['max_test_rp'].append(np.max(test_rps))
    dg_baseline_df['mean_val_rp'].append(np.mean(val_rps))
    dg_baseline_df['mean_test_rp'].append(np.mean(test_rps))
    dg_baseline_df['min_test_mae'].append(np.min(test_maes))
    dg_baseline_df['mean_test_mae'].append(np.mean(test_maes))
    dg_baseline_df['test_rp_std'].append(np.std(test_rps))
    dg_baseline_df['val_rp_std'].append(np.std(val_rps))
    dg_baseline_df['test_mae_std'].append(np.std(test_maes))

    # ── ML results ──
    for ml_name, ml_data in data['ml'].items():
        ml_fold_keys = [k for k in ml_data if k != 'aggregate']
        ml_val_rps  = [ml_data[k]['val']['rp']  for k in ml_fold_keys]
        ml_test_rps = [ml_data[k]['test']['rp'] for k in ml_fold_keys]
        ml_test_maes = [ml_data[k]['test']['mae'] for k in ml_fold_keys]

        dg_baseline_df['input'].append(f'{input_type}')
        dg_baseline_df['encoder_num_module'].append(None)
        dg_baseline_df['encoder_hdim'].append(None)
        dg_baseline_df['model'].append(ml_name)
        dg_baseline_df['max_val_rp'].append(np.max(ml_val_rps))
        dg_baseline_df['max_test_rp'].append(np.max(ml_test_rps))
        dg_baseline_df['mean_val_rp'].append(np.mean(ml_val_rps))
        dg_baseline_df['mean_test_rp'].append(np.mean(ml_test_rps))
        dg_baseline_df['min_test_mae'].append(np.min(ml_test_maes))
        dg_baseline_df['mean_test_mae'].append(np.mean(ml_test_maes))
        dg_baseline_df['test_rp_std'].append(np.std(ml_test_rps))
        dg_baseline_df['val_rp_std'].append(np.std(ml_val_rps))
        dg_baseline_df['test_mae_std'].append(np.std(ml_test_maes))

dg_baseline_df = pd.DataFrame(dg_baseline_df)

## ── Load pretrain+ML ΔG results ──

In [5]:
trained_data_root = '../GraPPI_data/trained_data_ssl_finetune/'
hidden_dims = [5,6,7,8,9, 10] #[5-10]
num_layers = [1,2,3,4,5,6] #[1-6]
input_types = ['base', 'esm'] #['base', 'esm']

dg_ml_df = {
    'input': [], 'encoder_num_module': [], 'encoder_hdim': [], 'model': [],
    'max_test_rp': [], 'max_val_rp': [], 'mean_val_rp': [], 'mean_test_rp': [], 'min_test_mae': [], 'mean_test_mae': [],
    'test_rp_std': [], 'val_rp_std': [], 'test_mae_std': []
}

for input_type in input_types:
    for hdim in hidden_dims:
        for nlayer in num_layers:
            if input_type == 'base':
                folder = f"{nlayer}layers_{hdim}hdim_jkmean" #jkconcat
            else:
                folder = f"{nlayer}layers_{hdim}hdim_{input_type}_jkmean" #jkconcat
            fpath = os.path.join(trained_data_root, folder, 'ml_dg_jkmean_5fold_results.json') #jkconcat
            if not os.path.exists(fpath):
                print(f"{fpath} not found")
                continue
            with open(fpath, 'r') as f:
                data = json.load(f)

            for ml_name, ml_data in data['ml'].items():
                fold_keys = [k for k in ml_data if k != 'aggregate']
                val_rps  = [ml_data[k]['val']['rp'] for k in fold_keys]
                test_rps = [ml_data[k]['test']['rp'] for k in fold_keys]
                test_maes = [ml_data[k]['test']['mae'] for k in fold_keys]

                dg_ml_df['input'].append(input_type)
                dg_ml_df['encoder_num_module'].append(nlayer)
                dg_ml_df['encoder_hdim'].append(2**hdim)
                dg_ml_df['model'].append(ml_name)
                dg_ml_df['max_val_rp'].append(np.max(val_rps))
                dg_ml_df['max_test_rp'].append(np.max(test_rps))
                dg_ml_df['mean_val_rp'].append(np.mean(val_rps))
                dg_ml_df['mean_test_rp'].append(np.mean(test_rps))
                dg_ml_df['min_test_mae'].append(np.min(test_maes))
                dg_ml_df['mean_test_mae'].append(np.mean(test_maes))
                dg_ml_df['test_rp_std'].append(np.std(test_rps))
                dg_ml_df['val_rp_std'].append(np.std(val_rps))
                dg_ml_df['test_mae_std'].append(np.std(test_maes))

dg_ml_df = pd.DataFrame(dg_ml_df)

## Combine all three

In [6]:
dg_cv_df_jk = pd.concat([dg_cv_df_jk, dg_ml_df, dg_baseline_df], ignore_index=True)
# change input column esm to GraPPI and base to GraPPI-base
dg_cv_df_jk['input'] = dg_cv_df_jk['input'].replace({'esm': 'GraPPI', 'base': 'GraPPI-base'})

#  mut ∆G prediction

## ── Load pretrain+MLP  mut ΔG results ──

In [7]:
trained_data_root = '../GraPPI_data/trained_data_ssl_finetune/'
hidden_dims = [5,6,7,8,9, 10] #[5-10]
num_layers = [1,2,3,4,5,6] #[1-6]
input_types = ['base', 'esm'] #['base', 'esm']
data_all = {}
for input_type in input_types:
	for hdim in hidden_dims:
		for nlayer in num_layers:
			if input_type == 'base':
				folder = f"{nlayer}layers_{hdim}hdim_jkmean"
			else:
				folder = f"{nlayer}layers_{hdim}hdim_{input_type}_jkmean"
			save_dir = os.path.join(trained_data_root, folder)
			if os.path.exists(f"{save_dir}/ddg_5fold_jkmean_training_log.json"):
				with open(f"{save_dir}/ddg_5fold_jkmean_training_log.json", 'r') as f:
					data_single = json.load(f)
				val_rps = [result_dict['best_val_rp'] for _,result_dict in data_single.items()]
				test_rps = [result_dict['test_rp'] for _,result_dict in data_single.items()]
				data_all[f"{folder}"] = {'test_rp': test_rps, 'val_rp': val_rps}
			else:
				print(f"{save_dir}/ddg_5fold_jkmean_training_log.json not found")
				continue

ddg_cv_df_jk = {}
nlayers, hdims, mean_val_rps, mean_test_rps, max_val_rps, max_test_rps, std_val_rps, std_test_rps, models, mean_test_maes, min_test_maes = [], [], [], [], [], [], [], [], [], [], []
for key, value in data_all.items():
	nlayer, hdim = int(key.split('_')[0].replace('layers','')), int(key.split('_')[1].replace('hdim',''))
	if len(key.split('_')) > 3:
		model = key.split('_')[2]
	else:
		model = 'base'
	models.append(model)
	nlayers.append(nlayer)
	hdims.append(2**hdim)
	mean_test_rps.append(np.mean(value['test_rp']))
	mean_val_rps.append(np.mean(value['val_rp']))
	max_test_rps.append(np.max(value['test_rp']))
	max_val_rps.append(np.max(value['val_rp']))
	std_test_rps.append(np.std(value['test_rp']))
	std_val_rps.append(np.std(value['val_rp']))
	mean_test_maes.append(np.mean(value['test_mae']) if 'test_mae' in value else np.nan)
	min_test_maes.append(np.min(value['test_mae']) if 'test_mae' in value else np.nan)

ddg_cv_df_jk['input'] = models
ddg_cv_df_jk['encoder_num_module'] = nlayers
ddg_cv_df_jk['encoder_hdim'] = hdims
ddg_cv_df_jk['model'] = 'MLP'
ddg_cv_df_jk['max_test_rp'] = max_test_rps
ddg_cv_df_jk['max_val_rp'] = max_val_rps
ddg_cv_df_jk['mean_val_rp'] = mean_val_rps
ddg_cv_df_jk['mean_test_rp'] = mean_test_rps
ddg_cv_df_jk['test_rp_std'] = std_test_rps
ddg_cv_df_jk['val_rp_std'] = std_val_rps
ddg_cv_df_jk['mean_test_mae'] = mean_test_maes
ddg_cv_df_jk['min_test_mae'] = min_test_maes
ddg_cv_df_jk = pd.DataFrame(ddg_cv_df_jk)

## ── Load baseline mut ΔG results ──

In [8]:
trained_data_root = '../GraPPI_data/trained_data_ssl_finetune/'

baseline_sources = {
    'base':   ('baseline',     'ddG_baseline_results.json'),
    'esm':    ('esm_baseline', 'ddG_esm_baseline_results.json'),
}

ddg_baseline_df = {
    'input': [], 'encoder_num_module': [], 'encoder_hdim': [],'model': [],
    'max_test_rp': [], 'max_val_rp': [], 'mean_val_rp': [], 'mean_test_rp': [], 'min_test_mae': [], 'mean_test_mae': [],
    'val_rp_std': [], 'test_rp_std': [], 'test_mae_std': []
}

for input_type, (subfolder, json_file) in baseline_sources.items():
    fpath = os.path.join(trained_data_root, subfolder, json_file)
    if not os.path.exists(fpath):
        print(f"{fpath} not found")
        continue
    with open(fpath, 'r') as f:
        data = json.load(f)

    # ── NN results ──
    nn = data['nn']
    fold_keys = [k for k in nn if k != 'aggregate']
    val_rps  = [nn[k]['best_val_rp'] for k in fold_keys]
    test_rps = [nn[k]['test']['rp']  for k in fold_keys]
    test_maes = [nn[k]['test']['mae']  for k in fold_keys]

    ddg_baseline_df['input'].append(f'{input_type}')
    ddg_baseline_df['encoder_num_module'].append(None)
    ddg_baseline_df['encoder_hdim'].append(None)
    ddg_baseline_df['model'].append('MLP')
    ddg_baseline_df['max_val_rp'].append(np.max(val_rps))
    ddg_baseline_df['max_test_rp'].append(np.max(test_rps))
    ddg_baseline_df['mean_val_rp'].append(np.mean(val_rps))
    ddg_baseline_df['mean_test_rp'].append(np.mean(test_rps))
    ddg_baseline_df['min_test_mae'].append(np.min(test_maes))
    ddg_baseline_df['mean_test_mae'].append(np.mean(test_maes))
    ddg_baseline_df['val_rp_std'].append(np.std(val_rps))
    ddg_baseline_df['test_rp_std'].append(np.std(test_rps))
    ddg_baseline_df['test_mae_std'].append(np.std(test_maes))

    # ── ML results ──
    for ml_name, ml_data in data['ml'].items():
        ml_fold_keys = [k for k in ml_data if k != 'aggregate']
        ml_val_rps  = [ml_data[k]['val']['rp']  for k in ml_fold_keys]
        ml_test_rps = [ml_data[k]['test']['rp'] for k in ml_fold_keys]
        ml_test_maes = [ml_data[k]['test']['mae'] for k in ml_fold_keys]

        ddg_baseline_df['input'].append(f'{input_type}')
        ddg_baseline_df['encoder_num_module'].append(None)
        ddg_baseline_df['encoder_hdim'].append(None)
        ddg_baseline_df['model'].append(ml_name)
        ddg_baseline_df['max_val_rp'].append(np.max(ml_val_rps))
        ddg_baseline_df['max_test_rp'].append(np.max(ml_test_rps))
        ddg_baseline_df['mean_val_rp'].append(np.mean(ml_val_rps))
        ddg_baseline_df['mean_test_rp'].append(np.mean(ml_test_rps))
        ddg_baseline_df['min_test_mae'].append(np.min(ml_test_maes))
        ddg_baseline_df['mean_test_mae'].append(np.mean(ml_test_maes))
        ddg_baseline_df['val_rp_std'].append(np.std(ml_val_rps))
        ddg_baseline_df['test_rp_std'].append(np.std(ml_test_rps))
        ddg_baseline_df['test_mae_std'].append(np.std(ml_test_maes))

ddg_baseline_df = pd.DataFrame(ddg_baseline_df)

## ── Load pretrain+ML mut ΔG results ──

In [9]:
trained_data_root = '../GraPPI_data/trained_data_ssl_finetune/'
hidden_dims = [5,6,7,8,9, 10] #[5-10]
num_layers = [1,2,3,4,5,6] #[1-6]
input_types = ['base', 'esm'] #['base', 'esm']

ddg_ml_df = {
    'input': [], 'encoder_num_module': [], 'encoder_hdim': [],'model': [],
    'max_test_rp': [], 'max_val_rp': [], 'mean_val_rp': [], 'mean_test_rp': [], 'min_test_mae': [], 'mean_test_mae': [],
    'test_rp_std': [], 'val_rp_std': [], 'test_mae_std': []
}

for input_type in input_types:
    for hdim in hidden_dims:
        for nlayer in num_layers:
            if input_type == 'base':
                folder = f"{nlayer}layers_{hdim}hdim_jkmean" #jkconcat
            else:
                folder = f"{nlayer}layers_{hdim}hdim_{input_type}_jkmean" #jkconcat
            fpath = os.path.join(trained_data_root, folder, 'ml_ddg_jkmean_5fold_results.json') #jkconcat
            if not os.path.exists(fpath):
                print(f"{fpath} not found")
                continue
            with open(fpath, 'r') as f:
                data = json.load(f)

            for ml_name, ml_data in data['ml'].items():
                fold_keys = [k for k in ml_data if k != 'aggregate']
                val_rps  = [ml_data[k]['val']['rp'] for k in fold_keys]
                test_rps = [ml_data[k]['test']['rp'] for k in fold_keys]
                test_maes = [ml_data[k]['test']['mae'] for k in fold_keys]

                ddg_ml_df['input'].append(f'{input_type}')
                ddg_ml_df['encoder_num_module'].append(nlayer)
                ddg_ml_df['encoder_hdim'].append(2**hdim)
                ddg_ml_df['model'].append(ml_name)
                ddg_ml_df['max_val_rp'].append(np.max(val_rps))
                ddg_ml_df['max_test_rp'].append(np.max(test_rps))
                ddg_ml_df['mean_val_rp'].append(np.mean(val_rps))
                ddg_ml_df['mean_test_rp'].append(np.mean(test_rps))
                ddg_ml_df['min_test_mae'].append(np.min(test_maes))
                ddg_ml_df['mean_test_mae'].append(np.mean(test_maes))
                ddg_ml_df['test_rp_std'].append(np.std(test_rps))
                ddg_ml_df['val_rp_std'].append(np.std(val_rps))
                ddg_ml_df['test_mae_std'].append(np.std(test_maes))

ddg_ml_df = pd.DataFrame(ddg_ml_df)

## Combine

In [10]:
ddg_cv_df_jk = pd.concat([ddg_cv_df_jk, ddg_ml_df, ddg_baseline_df], ignore_index=True)
# change input column esm to GraPPI and base to GraPPI-base
ddg_cv_df_jk['input'] = ddg_cv_df_jk['input'].replace({'esm': 'GraPPI', 'base': 'GraPPI-base'})

# Binder discrimination

## ── Load pretrain+MLP binder discrimination results ──

In [11]:
trained_data_root = '../GraPPI_data/trained_data_ssl_finetune/'
hidden_dims = [5,6,7,8,9, 10] #[5-10]
num_layers = [1,2,3,4,5,6] #[1-6]
input_types = ['base', 'esm'] #['base', 'esm']
bin_dis_df_jk = {
	'input': [], 'encoder_num_module': [], 'encoder_hdim': [], 'model': [],
	'test_auroc': [], 'test_f1': [], 'test_accuracy': [],
	'mean_unseen_auprc': [], 'mean_unseen_auroc': [],
    'unseen_auprc_std': [], 'unseen_auroc_std': [],
    'PosFix_mean_auroc': [], 'PosFix_mean_auprc': [],
    'PosFix_auprc_std': [], 'PosFix_auroc_std': []

}
for input_type in input_types:
	for hdim in hidden_dims:
		for nlayer in num_layers:
			if input_type == 'base':
				folder = f"{nlayer}layers_{hdim}hdim_jkmean"
			else:
				folder = f"{nlayer}layers_{hdim}hdim_{input_type}_jkmean"
			save_dir = os.path.join(trained_data_root, folder)
			suffix = f"jkmean"

			# Single-split (train/test) results
			log_path = f"{save_dir}/disc_binder_{suffix}_log.json"
			test_auroc = test_f1 = test_accuracy = np.nan
			if os.path.exists(log_path):
				with open(log_path, 'r') as f:
					data_single = json.load(f)
				test_auroc = max([sub_dict['auc'] for sub_dict in data_single['test_metrics']])
				test_accuracy = [sub_dict['accuracy'] for sub_dict in data_single['test_metrics'] if sub_dict['auc'] == test_auroc][0]
				test_f1 = [sub_dict['f1'] for sub_dict in data_single['test_metrics'] if sub_dict['auc'] == test_auroc][0]

			# 5-fold CV results (no held-out test set; val metrics aggregated across folds)
			cv_summary_path = f"{save_dir}/disc_binder_{suffix}_5fold_summary.json"
			cv_log_path     = f"{save_dir}/disc_binder_{suffix}_5fold_log.json"
			mean_unseen_auprc = mean_unseen_auroc = np.nan = unseen_auprc_std = unseen_auroc_std = np.nan
			if os.path.exists(cv_summary_path) and os.path.exists(cv_log_path):
				with open(cv_summary_path, 'r') as f:
					cv_summary = json.load(f)
				with open(cv_log_path, 'r') as f:
					cv_log = json.load(f)
				mean_unseen_auprc = cv_summary['mean_auprc_unseen']
				mean_unseen_auroc = cv_summary['mean_auc_unseen']
				unseen_auprc_std = cv_summary['std_auprc_unseen']
				unseen_auroc_std = cv_summary['std_auc_unseen']
			cv_summary_PosFix_path = f"{save_dir}/disc_binder_{suffix}_fixed_pos_5fold_summary.json" # e.g. disc_binder_jkmean_fixed_pos_5fold_summary.json
			cv_log_PosFix_path     = f"{save_dir}/disc_binder_{suffix}_fixed_pos_5fold_log.json"
			PosFix_auprc_std = PosFix_auroc_std = PosFix_mean_auprc = PosFix_mean_auroc = np.nan
			if os.path.exists(cv_summary_PosFix_path):
				with open(cv_summary_PosFix_path, 'r') as f:
					cv_summary_PosFix = json.load(f)
				PosFix_mean_auroc = cv_summary_PosFix['mean_auc_unseen']
				PosFix_mean_auprc = cv_summary_PosFix['mean_auprc_unseen']
				PosFix_auprc_std = cv_summary_PosFix['std_auprc_unseen']
				PosFix_auroc_std = cv_summary_PosFix['std_auc_unseen']

			bin_dis_df_jk['input'].append(input_type)
			bin_dis_df_jk['encoder_num_module'].append(nlayer)
			bin_dis_df_jk['encoder_hdim'].append(2**hdim)
			bin_dis_df_jk['model'].append('MLP')
			bin_dis_df_jk['test_auroc'].append(test_auroc)
			bin_dis_df_jk['test_accuracy'].append(test_accuracy)
			bin_dis_df_jk['test_f1'].append(test_f1)
			bin_dis_df_jk['mean_unseen_auprc'].append(mean_unseen_auprc)
			bin_dis_df_jk['mean_unseen_auroc'].append(mean_unseen_auroc)
			bin_dis_df_jk['unseen_auprc_std'].append(unseen_auprc_std)
			bin_dis_df_jk['unseen_auroc_std'].append(unseen_auroc_std)

			bin_dis_df_jk['PosFix_mean_auroc'].append(PosFix_mean_auroc)
			bin_dis_df_jk['PosFix_mean_auprc'].append(PosFix_mean_auprc)
			bin_dis_df_jk['PosFix_auprc_std'].append(PosFix_auprc_std)
			bin_dis_df_jk['PosFix_auroc_std'].append(PosFix_auroc_std)

bin_dis_df_jk = pd.DataFrame(bin_dis_df_jk)


## ── Load baseline binder discrimination results ──

In [12]:
trained_data_root = '../GraPPI_data/trained_data_ssl_finetune/'

baseline_sources = {
    'base':   ('baseline',     'disc_binder_baseline_results.json'),
    'esm':    ('esm_baseline', 'disc_binder_esm_baseline_results.json'),
}

bin_dis_baseline_df = {
    'input': [], 'encoder_num_module': [], 'encoder_hdim': [],'model':[],
    'test_auroc': [], 'test_f1': [], 'test_accuracy': [],
    'mean_unseen_auprc': [], 'mean_unseen_auroc': [],
    'unseen_auprc_std': [], 'unseen_auroc_std': [],
    'PosFix_mean_auroc': [], 'PosFix_mean_auprc': [],
    'PosFix_auprc_std': [], 'PosFix_auroc_std': []
}

for input_type, (subfolder, json_file) in baseline_sources.items():
    fpath = os.path.join(trained_data_root, subfolder, json_file)
    cv_glb_path = 'disc_binder_baseline_global_5fold_results.json' if input_type == 'base' else 'disc_binder_esm_baseline_global_5fold_results.json'
    cv_fp_path = 'disc_binder_baseline_fixed_pos_5fold_results.json' if input_type == 'base' else 'disc_binder_esm_baseline_fixed_pos_5fold_results.json'
    cv_fpath_global = os.path.join(trained_data_root, subfolder, cv_glb_path)
    cv_fpath_fix_pos = os.path.join(trained_data_root, subfolder, cv_fp_path)
    if not os.path.exists(fpath):
        print(f"{fpath} not found")
        continue
    with open(fpath, 'r') as f:
        data = json.load(f)
    if os.path.exists(cv_fpath_global):
        with open(cv_fpath_global, 'r') as f_cv:
            cv_data_global = json.load(f_cv)
            nn_global = cv_data_global['nn']
    if os.path.exists(cv_fpath_fix_pos):
        with open(cv_fpath_fix_pos, 'r') as f_cv:
            cv_data_fix_pos = json.load(f_cv)
        nn_fix_pos = cv_data_fix_pos['nn']
    # ── NN results (single split, not k-fold) ──
    nn = data['nn']
    bin_dis_baseline_df['input'].append(f'{input_type}')
    bin_dis_baseline_df['encoder_num_module'].append(None)
    bin_dis_baseline_df['encoder_hdim'].append(None)
    bin_dis_baseline_df['model'].append('MLP')
    bin_dis_baseline_df['test_auroc'].append(nn['test']['auc'])
    bin_dis_baseline_df['test_f1'].append(nn['test']['f1'])
    bin_dis_baseline_df['test_accuracy'].append(nn['test']['accuracy'])
    bin_dis_baseline_df['mean_unseen_auprc'].append(nn_global['aggregate'].get('auprc_unseen_mean',np.nan))
    bin_dis_baseline_df['mean_unseen_auroc'].append(nn_global['aggregate'].get('auc_unseen_mean',np.nan))
    bin_dis_baseline_df['unseen_auprc_std'].append(nn_global['aggregate'].get('auprc_unseen_std',np.nan))
    bin_dis_baseline_df['unseen_auroc_std'].append(nn_global['aggregate'].get('auc_unseen_std',np.nan))
    bin_dis_baseline_df['PosFix_mean_auroc'].append(nn_fix_pos['aggregate'].get('auc_unseen_mean',np.nan))
    bin_dis_baseline_df['PosFix_mean_auprc'].append(nn_fix_pos['aggregate'].get('auprc_unseen_mean',np.nan))
    bin_dis_baseline_df['PosFix_auprc_std'].append(nn_fix_pos['aggregate'].get('auprc_unseen_std',np.nan))
    bin_dis_baseline_df['PosFix_auroc_std'].append(nn_fix_pos['aggregate'].get('auc_unseen_std',np.nan))

    # ── ML results (single split, flat dict per model) ──
    for ml_name, ml_metrics in data['ml'].items():
        bin_dis_baseline_df['input'].append(f'{input_type}')
        bin_dis_baseline_df['encoder_num_module'].append(None)
        bin_dis_baseline_df['encoder_hdim'].append(None)
        bin_dis_baseline_df['model'].append(ml_name)
        bin_dis_baseline_df['test_auroc'].append(ml_metrics['auc'])
        bin_dis_baseline_df['test_f1'].append(ml_metrics['f1'])
        bin_dis_baseline_df['test_accuracy'].append(ml_metrics['accuracy'])
        
        bin_dis_baseline_df['mean_unseen_auprc'].append(cv_data_global['ml'].get(ml_name,{}).get('aggregate',{}).get('auprc_unseen_mean',np.nan))
        bin_dis_baseline_df['mean_unseen_auroc'].append(cv_data_global['ml'].get(ml_name,{}).get('aggregate',{}).get('auc_unseen_mean',np.nan))
        bin_dis_baseline_df['PosFix_mean_auroc'].append(cv_data_fix_pos['ml'].get(ml_name,{}).get('aggregate',{}).get('auc_unseen_mean',np.nan))
        bin_dis_baseline_df['PosFix_mean_auprc'].append(cv_data_fix_pos['ml'].get(ml_name,{}).get('aggregate',{}).get('auprc_unseen_mean',np.nan))
        
        bin_dis_baseline_df['unseen_auprc_std'].append(cv_data_global['ml'].get(ml_name,{}).get('aggregate',{}).get('auprc_unseen_std',np.nan))
        bin_dis_baseline_df['unseen_auroc_std'].append(cv_data_global['ml'].get(ml_name,{}).get('aggregate',{}).get('auc_unseen_std',np.nan))
        bin_dis_baseline_df['PosFix_auprc_std'].append(cv_data_fix_pos['ml'].get(ml_name,{}).get('aggregate',{}).get('auprc_unseen_std',np.nan))
        bin_dis_baseline_df['PosFix_auroc_std'].append(cv_data_fix_pos['ml'].get(ml_name,{}).get('aggregate',{}).get('auc_unseen_std',np.nan))

bin_dis_baseline_df = pd.DataFrame(bin_dis_baseline_df)

## ── Load pretrain+ML binder discrimination results ──

In [13]:
trained_data_root = '../GraPPI_data/trained_data_ssl_finetune/'
hidden_dims = [5,6,7,8,9, 10] #[5-10]
num_layers = [1,2,3,4,5,6] #[1-6]
input_types = ['base', 'esm'] #['base', 'esm']

bin_dis_ml_df = {
    'input': [], 'encoder_num_module': [], 'encoder_hdim': [],'model': [],
    'test_auroc': [], 'test_f1': [], 'test_accuracy': [],
    'mean_unseen_auprc': [], 'mean_unseen_auroc': [],
    'PosFix_mean_auroc': [], 'PosFix_mean_auprc': []
}

for input_ in input_types:
    for hdim in hidden_dims:
        for nlayer in num_layers:
            if input_ == 'base':
                folder = f"{nlayer}layers_{hdim}hdim_jkmean" #jkconcat
            else:
                folder = f"{nlayer}layers_{hdim}hdim_{input_}_jkmean" #jkconcat
            fpath = os.path.join(trained_data_root, folder, 'ml_disc_binder_jkmean_results.json') #jkconcat
            if not os.path.exists(fpath):
                print(f"{fpath} not found")
                continue
            with open(fpath, 'r') as f:
                data = json.load(f)

            for ml_name, ml_data in data['ml'].items():
                bin_dis_ml_df['input'].append(f'{input_}')
                bin_dis_ml_df['encoder_num_module'].append(nlayer)
                bin_dis_ml_df['encoder_hdim'].append(2**hdim)
                bin_dis_ml_df['model'].append(ml_name)
                bin_dis_ml_df['test_auroc'].append(ml_data['test']['auc'])
                bin_dis_ml_df['test_f1'].append(ml_data['test']['f1'])
                bin_dis_ml_df['test_accuracy'].append(ml_data['test']['accuracy'])
                bin_dis_ml_df['mean_unseen_auprc'].append(np.nan)
                bin_dis_ml_df['mean_unseen_auroc'].append(np.nan)
                bin_dis_ml_df['PosFix_mean_auroc'].append(np.nan)
                bin_dis_ml_df['PosFix_mean_auprc'].append(np.nan)

bin_dis_ml_df = pd.DataFrame(bin_dis_ml_df)


## Combine

In [14]:
bin_dis_df_jk = pd.concat([bin_dis_df_jk, bin_dis_ml_df, bin_dis_baseline_df], ignore_index=True)
# change input column esm to GraPPI and base to GraPPI-base
bin_dis_df_jk['input'] = bin_dis_df_jk['input'].replace({'esm': 'GraPPI', 'base': 'GraPPI-base'})

# Show tables

In [15]:
ssl_df.sort_values(by='test_loss', ascending=True).head(20)

,input_feat,encoder_num_module,encoder_hdim,test_loss
35,GraPPI-base,6,1024,0.123967
29,GraPPI-base,6,512,0.124095
34,GraPPI-base,5,1024,0.126931
65,GraPPI,6,512,0.128568
56,GraPPI,3,256,0.129820
63,GraPPI,4,512,0.130576
70,GraPPI,5,1024,0.131331
64,GraPPI,5,512,0.132027
69,GraPPI,4,1024,0.132482
57,GraPPI,4,256,0.133444


In [16]:
ssl_df.to_csv('./result_csv_data/SSL_loss.csv',index=False)

In [17]:
bin_dis_df_jk[bin_dis_df_jk['mean_unseen_auprc'].notnull()].sort_values(by=['mean_unseen_auprc'],ascending=False).head(50) # [(bin_dis_df_jk['input']=='base') & (bin_dis_df_jk['model']=='MLP')]

,input,encoder_num_module,encoder_hdim,model,test_auroc,test_f1,test_accuracy,mean_unseen_auprc,mean_unseen_auroc,unseen_auprc_std,unseen_auroc_std,PosFix_mean_auroc,PosFix_mean_auprc,PosFix_auprc_std,PosFix_auroc_std
68,GraPPI,3,1024,MLP,0.898540,0.856668,0.851101,0.776102,0.931952,0.007237,0.005734,0.933209,0.925022,0.003368,0.001475
67,GraPPI,2,1024,MLP,0.909206,0.854779,0.848511,0.767413,0.932281,0.011690,0.005399,0.935452,0.925887,0.006712,0.004059
69,GraPPI,4,1024,MLP,0.902097,0.848356,0.845706,0.765447,0.927992,0.020913,0.005566,0.930798,0.925925,0.003909,0.002113
70,GraPPI,5,1024,MLP,0.889344,0.848837,0.842900,0.748617,0.922920,0.011093,0.002822,0.924394,0.918064,0.008386,0.005244
63,GraPPI,4,512,MLP,0.883858,0.835728,0.829737,0.721110,0.908964,0.012573,0.009786,0.915703,0.908012,0.005461,0.004575
66,GraPPI,1,1024,MLP,0.887230,0.846519,0.839663,0.719656,0.913271,0.023200,0.005617,0.924309,0.913573,0.004915,0.002788
64,GraPPI,5,512,MLP,0.867099,0.807542,0.808373,0.718013,0.908173,0.019099,0.003690,0.900079,0.894059,0.006817,0.003415
71,GraPPI,6,1024,MLP,0.859374,0.775631,0.787009,0.716124,0.909310,0.011909,0.004380,0.882886,0.878929,0.009941,0.008317
62,GraPPI,3,512,MLP,0.893504,0.838481,0.832974,0.703139,0.904518,0.017401,0.004543,0.916599,0.908499,0.006237,0.005544
61,GraPPI,2,512,MLP,0.882149,0.825330,0.820242,0.699443,0.905017,0.024428,0.004487,0.903585,0.897322,0.005562,0.005984


In [18]:
bin_dis_df_jk.to_csv('./result_csv_data/binder_cls.csv',index=False)

In [25]:
dg_cv_df_jk[(dg_cv_df_jk['input']=='GraPPI')&(dg_cv_df_jk['model']=='SVR')].sort_values(by=['max_test_rp'],ascending=False).head(40) #[dg_cv_df_jk['input']=='base']

,input,encoder_num_module,encoder_hdim,model,max_test_rp,max_val_rp,mean_val_rp,mean_test_rp,min_test_mae,mean_test_mae,test_rp_std,val_rp_std,test_mae_std
335,GraPPI,6,512,SVR,0.684564,0.683083,0.658409,0.645467,1.572685,1.603639,0.021086,0.017045,0.019382
331,GraPPI,5,512,SVR,0.662126,0.665320,0.641374,0.639962,1.549314,1.588233,0.018797,0.016152,0.028492
355,GraPPI,5,1024,SVR,0.658340,0.675017,0.652690,0.629343,1.588467,1.611501,0.018281,0.014218,0.018878
351,GraPPI,4,1024,SVR,0.657283,0.678110,0.654920,0.636398,1.568257,1.593982,0.011548,0.014441,0.020220
311,GraPPI,6,256,SVR,0.657281,0.651993,0.631783,0.640303,1.589353,1.606135,0.009882,0.015450,0.014250
303,GraPPI,4,256,SVR,0.653124,0.644888,0.631182,0.631702,1.577678,1.598148,0.013036,0.014624,0.016571
307,GraPPI,5,256,SVR,0.651947,0.645030,0.623378,0.617106,1.637107,1.663085,0.018138,0.015217,0.022461
347,GraPPI,3,1024,SVR,0.649970,0.676744,0.653972,0.623144,1.610137,1.623674,0.015678,0.015034,0.007814
359,GraPPI,6,1024,SVR,0.647848,0.682286,0.661312,0.614819,1.606829,1.633774,0.020590,0.013796,0.024161
315,GraPPI,1,512,SVR,0.644294,0.678605,0.656960,0.608149,1.569015,1.621481,0.018662,0.017433,0.029912


In [20]:
dg_cv_df_jk.to_csv('result_csv_data/dG_pred.csv',index=False)

In [21]:
ddg_cv_df_jk[(ddg_cv_df_jk['input']=='GraPPI')&(ddg_cv_df_jk['model']=='SVR')].sort_values(by=['max_test_rp'],ascending=False).head(30) #[ddg_cv_df_jk['input']=='base']

,input,encoder_num_module,encoder_hdim,model,max_test_rp,max_val_rp,mean_val_rp,mean_test_rp,test_rp_std,val_rp_std,mean_test_mae,min_test_mae,test_mae_std
369,GraPPI,None,None,SVR,0.846798,0.850335,0.823563,0.837523,0.007309,0.015310,1.352723,1.269742,0.041958
343,GraPPI,2,1024,SVR,0.837668,0.878510,0.858925,0.817057,0.016377,0.011227,1.297715,1.197919,0.060200
275,GraPPI,3,128,SVR,0.816431,0.855036,0.828047,0.772428,0.027518,0.016380,1.265003,1.179415,0.046834
287,GraPPI,6,128,SVR,0.787732,0.857949,0.828877,0.746065,0.029661,0.016609,1.399872,1.346788,0.046807
323,GraPPI,3,512,SVR,0.778258,0.875524,0.852907,0.708440,0.036219,0.012802,1.593637,1.437787,0.082896
283,GraPPI,5,128,SVR,0.775748,0.851979,0.826027,0.747647,0.024911,0.015695,1.356979,1.310770,0.037915
347,GraPPI,3,1024,SVR,0.765613,0.875238,0.855025,0.724268,0.025538,0.011373,1.599738,1.491523,0.084181
335,GraPPI,6,512,SVR,0.751355,0.875810,0.855196,0.668482,0.045130,0.012162,1.608311,1.471900,0.089922
291,GraPPI,1,256,SVR,0.748050,0.872882,0.852007,0.691892,0.040759,0.012229,1.589852,1.523162,0.077916
227,GraPPI,3,32,SVR,0.728388,0.805706,0.772608,0.720178,0.007693,0.022908,1.364950,1.332695,0.021493


In [22]:
ddg_cv_df_jk.to_csv('result_csv_data/mut_dG_pred.csv',index=False)